# AI301 Assignment #2 — EDA (Cats and Dogs++)

All exploratory data analysis in one place, built to run on **Kaggle** with the competition data attached (no GPU needed).

**Competition constraints that shape the analysis:**
- **Task:** multi-label, 5 binary labels per image (Cat, Dog, Moris, Motya, Biatrix); background images have none.
- **Metric: macro F1**, so every class counts 20%. The three rare identity classes are 60% of the score.
- **Data:** no external data. The public rules forbid manual relabelling, but the **competition owner confirmed that fixing train labels is allowed**
  (test is never touched).
- **Weights:** any pretrained weights from `timm`; from other sources, ImageNet weights only.

Checks: data format, class imbalance, labels per image, label co-occurrence, image size / quality and train-vs-test shift,
outliers, near-duplicates (train–train, train–test, test–test), the same image checks on test,
a full review of the rare-class train images, an ImageNet-classifier label check, hand-checked train label fixes, scene groups + the validation split, and a summary of the modelling decisions.
At the end the cleaned train table (label fixes + deduplication + `scene_group` + `is_val`) is saved to `/kaggle/working/train_clean.csv` for the training notebooks.

## 1. Setup

In [ ]:
import os, random, warnings
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.fft import dctn
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

### Plot helpers
`label_code`, `label_names`, `pet_labels` and `show_grid` (inlined here so the notebook does not depend on a local package).

In [ ]:
def label_code(r, labels):
    """0/1 string of a row's labels in `labels` order, e.g. '10010'."""
    return "".join(str(int(r[l])) for l in labels)

def label_names(r, labels):
    """Comma-separated positive labels of a row, or 'none'."""
    return ", ".join(l for l in labels if r[l] == 1) or "none"

def pet_labels(r):
    """Labels as text, species first: 'Cat: Moris, Motya | Dog: Biatrix', 'Cat' (no named cat), or 'none'."""
    parts = []
    for species, names in [("Cat", ["Moris", "Motya"]), ("Dog", ["Biatrix"])]:
        named = [name for name in names if r[name] == 1]
        if r[species] == 1 or named:
            parts.append(f"{species}: {', '.join(named)}" if named else species)
    return " | ".join(parts) or "none"

def show_grid(rows, labels=None, title=None, suptitle=None, ncols=None, size=2.6):
    """One subplot row per DataFrame in `rows`, one image (column `path`) per cell.
    `title(row)` sets each image's title; by default the label names and code (if `labels` is given) or the image_id."""
    if title is None:
        title = (lambda r: f"{label_names(r, labels)} | {label_code(r, labels)}") if labels else (lambda r: str(r.image_id)[:8])
    rows = [df for df in rows if len(df)]
    ncols = ncols or max(len(df) for df in rows)
    fig, axes = plt.subplots(len(rows), ncols, figsize=(size * ncols, size * len(rows)), squeeze=False)
    for ax_row, df in zip(axes, rows):
        for ax in ax_row:
            ax.axis("off")
        for ax, (_, r) in zip(ax_row, df.iterrows()):
            with Image.open(r.path) as im:
                ax.imshow(im.convert("RGB"))
            ax.set_title(title(r), fontsize=7)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()

## 2. Locate the data

In [ ]:
# Find the competition folder (the one containing train.csv). Kaggle may mount the data through symlinks, which
# Path.rglob does not follow (Python < 3.13), so: known path first, then a search that follows symlinks.
import glob
KNOWN_CSV = Path("/kaggle/input/competitions/cats-and-dogs/train.csv")
if KNOWN_CSV.exists():
    train_csv = KNOWN_CSV
else:
    found = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    if not found:
        found = [os.path.join(root, "train.csv") for root, _, files in os.walk("/kaggle/input", followlinks=True) if "train.csv" in files]
    print("/kaggle/input contains:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "nothing")
    assert found, "train.csv not found under /kaggle/input, attach the competition data"
    train_csv = Path(found[0])
DATA_DIR = train_csv.parent
OUT_DIR = Path("/kaggle/working")
print("DATA_DIR:", DATA_DIR)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

LABELS = sample_sub.columns.drop("image_id").tolist()
print("labels:", LABELS)
print("train", train.shape, "| test", test.shape, "| sample_submission", sample_sub.shape)

In [ ]:
# Map image_id -> file path (extension / folder structure is not assumed)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
id2path = {}
for root, _, files in os.walk(DATA_DIR, followlinks=True):   # follow symlinked image folders
    for f in files:
        p = Path(f)
        if p.suffix.lower() in IMG_EXT:
            id2path[p.stem] = os.path.join(root, f)
print("image files found:", len(id2path))

train["path"] = train.image_id.map(id2path)
test["path"] = test.image_id.map(id2path)
print("missing train images:", train.path.isna().sum(), "| missing test images:", test.path.isna().sum())
assert train.path.notna().all() and test.path.notna().all(), "Some images are missing, check the folder layout"

## 3. Train / test / submission format

In [ ]:
print("--- train.csv (example rows) ---")
display(train.drop(columns="path").head())

print("--- test.csv (example rows) ---")
display(test.drop(columns="path").head())

print("--- sample_submission.csv (example rows) ---")
display(sample_sub.head())

print("Duplicate image_ids in train:", train.image_id.duplicated().sum())
print("Test ids == sample_submission ids (same order):", (test.image_id.values == sample_sub.image_id.values).all())
print("Sample-submission values:", sorted(np.unique(sample_sub[LABELS].values)),
      "-> hard 0/1 predictions are expected (check the Kaggle 'Evaluation' tab)")

## 4. Labels
This is a **multi-label** problem (5 independent binary columns), so we check: class prevalence, how many labels per image,
label co-occurrence, and what the classes look like.

In [ ]:
# Class prevalence: how many positives each label has
counts = train[LABELS].sum().sort_values(ascending=False)

prevalence = pd.DataFrame({
    "positives": counts,
    "prevalence_%": (counts / len(train) * 100).round(2),
    "neg:pos ratio": ((len(train) - counts) / counts).round(1),
})
display(prevalence)

rare = counts[counts < 0.02 * len(train)].index.tolist()
print("Severely under-represented classes (<2% of train):", rare)

In [ ]:
# Positives per class (log scale, so the rare classes stay visible)
ax = counts.plot.bar(figsize=(6, 3.5), color="steelblue", logy=True, title="Positives per class")
ax.bar_label(ax.containers[0])
plt.tight_layout()
plt.show()

In [ ]:
samples = [train[train[label] == 1].sample(min(10, int(train[label].sum())), random_state=SEED) for label in LABELS]
show_grid(samples, LABELS, suptitle="Example images per class (one row per label)")

In [ ]:
# How many positive labels each image has (0 = background / other)
n_labels = train[LABELS].sum(axis=1)

ax = n_labels.value_counts().sort_index().plot.bar(
    figsize=(6, 3.5), color="indianred", logy=True, title="Number of positive labels per image"
)
ax.bar_label(ax.containers[0])
plt.tight_layout()
plt.show()

print("Images with NO positive label:", int((n_labels == 0).sum()),
      "-> a valid 'background / other' case, the model must be allowed to predict all zeros")

In [ ]:
samples = [group.sample(min(5, len(group)), random_state=SEED) for _, group in train.groupby(n_labels)]
show_grid(samples, LABELS, suptitle="Example images by number of positive labels (row k = k labels)")

In [ ]:
co = train[LABELS].T.dot(train[LABELS])
k = len(LABELS)
plt.figure(figsize=(5, 4))
plt.imshow(np.log1p(co.values), cmap="Blues")
plt.xticks(range(k), LABELS, rotation=45)
plt.yticks(range(k), LABELS)
for i in range(k):
    for j in range(k):
        plt.text(j, i, int(co.values[i, j]), ha="center", va="center", fontsize=8)
plt.title("Co-occurrence (colour = log count)")
plt.tight_layout()
plt.show()

combo = train[LABELS].astype(str).agg("".join, axis=1).value_counts()
display(combo.rename("count").to_frame().assign(pattern=lambda d: d.index).head(12))
print("Pattern order:", LABELS)

## 5. Image size & quality: train vs test (code commented out for speed)
Every image is checked (not a sample). Quality stats are computed on a 256×256 copy so they are comparable across images.
- **w, h, aspect**: original size
- **brightness**: mean grey level (0 = black, 1 = white)
- **contrast**: std of grey levels (≈0 = flat / blank image)
- **saturation**: mean (max − min) over RGB channels (≈0 = grayscale)
- **sharpness**: log10 variance of the Laplacian (low = blurry)

In [ ]:
# Commented out for speed (image statistics over all ~31k images). To re-enable: select the cell and press Cmd+/.
# LUMA = np.array([0.299, 0.587, 0.114], dtype=np.float32)   # RGB -> grey weights
#
# def img_stats(path):
#     with Image.open(path) as im:
#         w, h = im.size
#         im.draft("RGB", (256, 256))   # let the JPEG decoder skip full resolution (much faster)
#         rgb = np.asarray(im.convert("RGB").resize((256, 256)), dtype=np.float32) / 255
#
#     gray = rgb @ LUMA
#     laplacian = gray[:-2, 1:-1] + gray[2:, 1:-1] + gray[1:-1, :-2] + gray[1:-1, 2:] - 4 * gray[1:-1, 1:-1]
#     return dict(
#         w=w,
#         h=h,
#         aspect=w / h,
#         brightness=gray.mean(),
#         contrast=gray.std(),
#         saturation=(rgb.max(axis=2) - rgb.min(axis=2)).mean(),
#         sharpness=np.log10(laplacian.var() + 1e-8),
#     )
#
# with Pool() as pool:
#     tr_s = pd.DataFrame(pool.map(img_stats, train.path, chunksize=64))
#     te_s = pd.DataFrame(pool.map(img_stats, test.path, chunksize=64))
#
# print("Image sizes (w, h) in train:", tr_s.value_counts(["w", "h"]).head(10).to_dict())
# print("Image sizes (w, h) in test: ", te_s.value_counts(["w", "h"]).head(10).to_dict())
# print("Distinct (w, h) — train:", tr_s[["w", "h"]].drop_duplicates().shape[0],
#       "| test:", te_s[["w", "h"]].drop_duplicates().shape[0])
# display(pd.concat({"train": tr_s.describe().T, "test": te_s.describe().T}, axis=1).round(3))

In [ ]:
# Commented out for speed (image statistics over all ~31k images). To re-enable: select the cell and press Cmd+/.
# PLOT_STATS = ["w", "aspect", "brightness", "contrast", "saturation", "sharpness"]
#
# fig, axes = plt.subplots(2, 3, figsize=(17, 7))
# for ax, col in zip(axes.ravel(), PLOT_STATS):
#     # Clip the x-range to the 0.1-99.9 percentiles so a few extreme images don't squash the plot
#     lo, hi = np.percentile(pd.concat([tr_s[col], te_s[col]]), [0.1, 99.9])
#     bins = np.linspace(lo, hi + 1e-6, 50)
#     ax.hist(tr_s[col], bins=bins, alpha=0.6, density=True, label="train")
#     ax.hist(te_s[col], bins=bins, alpha=0.6, density=True, label="test")
#     ax.set_title(col)
#     ax.legend()
# fig.suptitle("Image size & quality stats: train vs test")
# plt.tight_layout()
# plt.show()

### Quality flags (threshold-based)

In [ ]:
# Commented out for speed (image statistics over all ~31k images). To re-enable: select the cell and press Cmd+/.
# # name: (stat, direction, threshold). Thresholds pick roughly the most extreme ~1% of images
# QUALITY_CHECKS = {
#     "grayscale": ("saturation", "<", 0.02),
#     "very dark": ("brightness", "<", 0.15),
#     "very bright": ("brightness", ">", 0.85),
#     "low contrast": ("contrast", "<", 0.08),
#     "blurry": ("sharpness", "<", -3.5),
# }
#
# def is_flagged(stats, stat, direction, threshold):
#     return stats[stat] < threshold if direction == "<" else stats[stat] > threshold
#
# summary = pd.DataFrame({
#     name: {
#         "train": is_flagged(tr_s, *check).sum(),
#         "test": is_flagged(te_s, *check).sum(),
#         "train_%": is_flagged(tr_s, *check).mean() * 100,
#         "test_%": is_flagged(te_s, *check).mean() * 100,
#     }
#     for name, check in QUALITY_CHECKS.items()
# }).T
# display(summary.round(2))

In [ ]:
# Commented out for speed (image statistics over all ~31k images). To re-enable: select the cell and press Cmd+/.
# def most_extreme(df, stats, n=10):
#     """One DataFrame per quality check: the n most extreme images, with a `reason` column for the title."""
#     df = df.join(stats)
#     rows = []
#     for name, (stat, direction, _) in QUALITY_CHECKS.items():
#         worst = df.sort_values(stat, ascending=(direction == "<")).head(n)
#         rows.append(worst.assign(reason=name + " " + worst[stat].round(2).astype(str)))
#     return rows
#
# checks = ", ".join(QUALITY_CHECKS)
# show_grid(most_extreme(train, tr_s), title=lambda r: f"{r.reason} | {label_names(r, LABELS)} | {label_code(r, LABELS)}",
#           suptitle=f"Most extreme train images per check (rows: {checks})")
# # Test has no labels, so show the start of the image_id instead
# show_grid(most_extreme(test, te_s), title=lambda r: f"{r.reason} | {r.image_id[:8]}",
#           suptitle=f"Most extreme test images per check (rows: {checks})")

### Size / brightness outliers (IQR rule)
An image is an outlier candidate when its width, height, aspect or brightness lies more than 3 IQR outside the quartiles.

In [ ]:
# Commented out for speed (image statistics over all ~31k images). To re-enable: select the cell and press Cmd+/.
# def iqr_flags(s, k=3.0):
#     q1, q3 = s.quantile([.25, .75])
#     iqr = q3 - q1
#     return (s < q1 - k * iqr) | (s > q3 + k * iqr)
#
# flags = pd.concat([iqr_flags(tr_s[c]) for c in ["w", "h", "aspect", "brightness"]], axis=1).any(axis=1)
# print(f"Train outlier candidates: {flags.sum()} / {len(train)}")
# out = train[flags.values].join(tr_s).head(10)
# if len(out):
#     show_grid([out], title=lambda r: f"{int(r.w)}x{int(r.h)} | {label_code(r, LABELS)}",
#               suptitle="Train outlier candidates (first 10)", ncols=5 if len(out) <= 5 else 10)

## 6. Near-duplicates (train–train, train–test, test–test)
Each image gets a 64-bit **perceptual hash** (pHash): the signs of the lowest DCT frequencies of a 32×32 grey thumbnail.
Re-encoded, slightly resized or recoloured copies of a photo get (almost) the same hash. We call two images duplicates
when their hashes differ in at most `MAX_HASH_DIST` bits. Larger distances start matching *different* objects on a plain
background, so the threshold is kept tight.

Why it matters: duplicates split across train/val inflate CV, and train–test duplicates tell how much the leaderboard rewards memorisation.

In [ ]:
def phash(path):
    """64-bit perceptual hash: is each low-frequency DCT coefficient above the median?"""
    with Image.open(path) as im:
        gray = np.asarray(im.convert("L").resize((32, 32), Image.Resampling.LANCZOS), dtype=np.float32)
    low = dctn(gray, norm="ortho")[:8, :8].ravel()[1:]   # 63 lowest frequencies, without the mean (DC) term
    bits = np.append(low > np.median(low), False)         # pad to 64 bits
    return np.packbits(bits).view(">u8")[0]

all_imgs = pd.concat([train.assign(split="train"), test.assign(split="test")], ignore_index=True)
with Pool() as pool:
    hashes = np.array(pool.map(phash, all_imgs.path, chunksize=64), dtype=np.uint64)

In [ ]:
MAX_HASH_DIST = 2

# Hamming distance = popcount of XOR. np.bitwise_count needs numpy >= 2.0, so fall back to a 16-bit lookup table
if hasattr(np, "bitwise_count"):
    popcount = np.bitwise_count
else:
    _POP16 = np.array([bin(i).count("1") for i in range(1 << 16)], dtype=np.uint8)
    def popcount(x):
        x = x.astype(np.uint64)
        return sum(_POP16[(x >> np.uint64(s)) & np.uint64(0xFFFF)] for s in (0, 16, 32, 48))

# Compare every image with every other one, in chunks to keep memory small
pair_i, pair_j = [], []
for start in range(0, len(hashes), 1000):
    dist = popcount(hashes[start:start + 1000, None] ^ hashes[None, :])
    i, j = np.nonzero(dist <= MAX_HASH_DIST)
    i += start
    keep = i < j                                                                     # each pair once, no self-pairs
    pair_i.append(i[keep])
    pair_j.append(j[keep])
pairs = pd.DataFrame({"i": np.concatenate(pair_i), "j": np.concatenate(pair_j)})
pairs["kind"] = all_imgs.split[pairs.i].values + "-" + all_imgs.split[pairs.j].values

# Duplicate groups = connected components of the "is duplicate" graph (singletons get their own group)
n = len(all_imgs)
graph = coo_matrix((np.ones(len(pairs)), (pairs.i, pairs.j)), shape=(n, n))
_, all_imgs["dup_group"] = connected_components(graph, directed=False)
train["dup_group"] = all_imgs.dup_group[:len(train)].values   # same row order as train.csv

group_size = all_imgs.dup_group.map(all_imgs.dup_group.value_counts())
test_groups = all_imgs.dup_group[all_imgs.split == "test"]
print("Duplicate pairs by kind:", pairs.kind.value_counts().to_dict())
print("Largest duplicate group:", group_size.max(), "images")
print(f"Train images with a duplicate in train: {train.dup_group.duplicated(keep=False).sum()} / {len(train)}")
print(f"Test images with a duplicate in train:  {test_groups.isin(train.dup_group).sum()} / {len(test)}")

In [ ]:
# Do duplicates inside train agree on their labels?
train_code = train[LABELS].astype(str).agg("".join, axis=1)
dup_train = train[train.dup_group.duplicated(keep=False)]
n_codes = train_code[dup_train.index].groupby(dup_train.dup_group).nunique()
print(f"Train duplicate groups: {len(n_codes)}, with conflicting labels: {(n_codes > 1).sum()}")

# How many rare-class positives have a duplicate (in train or test)?
for label in rare:
    pos = train[train[label] == 1]
    print(f"{label}: {len(pos)} positives, {(group_size[pos.index] > 1).sum()} have a duplicate")

In [ ]:
def dup_title(r):
    return f"{r.split} {r.image_id[:8]} | {label_code(r, LABELS)}" if r.split == "train" else f"{r.split} {r.image_id[:8]}"

# One grid row per kind, up to 3 example pairs side by side
rows = []
for kind in ["train-train", "train-test", "test-test"]:
    kind_pairs = pairs[pairs.kind == kind]
    if len(kind_pairs) == 0:
        print(f"no {kind} duplicates")
        continue
    examples = kind_pairs.sample(min(3, len(kind_pairs)), random_state=SEED)
    rows.append(all_imgs.loc[np.ravel(examples[["i", "j"]].values)])
if rows:
    show_grid(rows, title=dup_title, suptitle="Duplicate pairs (rows: train-train, train-test, test-test)", ncols=6)

## 7. Test set: same checks as train (stats check commented out for speed)
Test has no labels, so only the image-level checks can be repeated. Size, quality stats, quality flags, most extreme images (section 5)
and duplicates (section 6) already cover both sets. Here: the IQR outliers for test, test-only blank images, and a random look at test.

In [ ]:
# Commented out for speed (image statistics over all ~31k images). To re-enable: select the cell and press Cmd+/.
# # Same IQR rule as for train (section 5)
# te_flags = pd.concat([iqr_flags(te_s[c]) for c in ["w", "h", "aspect", "brightness"]], axis=1).any(axis=1)
# print(f"Outlier candidates — train: {flags.sum()} / {len(train)} | test: {te_flags.sum()} / {len(test)}")
# out = test[te_flags.values].join(te_s).head(10)
# if len(out):
#     show_grid([out], title=lambda r: f"{int(r.w)}x{int(r.h)} | b={r.brightness:.2f} | {r.image_id[:8]}",
#               suptitle="Test outlier candidates (first 10)", ncols=max(5, len(out)))
#
# # Blank / constant images (contrast ~0): none in train (min contrast 0.04)
# blank = te_s.contrast < 0.01
# print(f"Blank images — train: {(tr_s.contrast < 0.01).sum()} | test: {blank.sum()}", test.image_id[blank.values].tolist())

In [ ]:
show_grid([test.sample(10, random_state=s) for s in range(3)], suptitle="Random test images")

## 8. Rare-class review (train) was done and only one picture were relabled in the further section

## 9. Label check with an ImageNet classifier (train) — done, results below
**Done once on Kaggle (GPU); the model is no longer run here.** The confirmed fixes are hard-coded in section 10.

**Method.** A strong ImageNet classifier (`convnext_base.fb_in22k_ft_in1k` from `timm`, allowed by the rules) scored every train image
(one per duplicate pair, image + horizontal flip averaged, whole image resized without crop):
**p_cat** = summed probability of the 5 domestic-cat classes (ImageNet 281–285), **p_dog** = summed probability of the 118 dog breeds (151–268).
Candidates were listed most suspicious first and **every one was checked by eye**; the model only proposed, a person decided.

| Check | Rule | Candidates | Confirmed mistakes |
|---|---|---|---|
| Cat label, but no cat found | Cat = 1 and p_cat < 0.10 | 302 | 5 (1 was already dropped in the rare-class review) |
| Dog label, but no dog found | Dog = 1 and p_dog < 0.10 | 15 | 5 (1 was already fixed: the "Adopted" placeholder) |
| No Cat label, but a cat found | Cat = 0 and p_cat > 0.90 | 0 | – |
| No Dog label, but a dog found | Dog = 0 and p_dog > 0.90 | 6 | 2 |

**Why so many false alarms (302 → 5):** most flagged "Cat" photos do show a cat, but small, partly hidden, in a box / basket / sink,
or as a lynx-like breed; ImageNet then names the object (hamper, bucket, dishwasher, carton, …) instead. Dogs labelled Cat that ImageNet
calls a dog breed are mostly black long-haired cats (schipperke, groenendael) — correct labels. This is why every candidate was reviewed by hand
and nothing was relabelled automatically. Species only: the identity labels (Moris / Motya / Biatrix) were not checked by the model.

## 10. Label fixes and train deduplication (train only)
Hand-checked wrong train labels and unusable train images (fixing train labels was confirmed as allowed by the competition owner),
then drop train–train duplicates found by the pHash check in section 6, keeping the first image of each group.
Train only, never test. The EDA above shows the original data.

In [ ]:
# (image_id, labels after the fix, reason). Found by: [R] rare-class review (section 8), [I] ImageNet label check (section 9)
LABEL_FIXES = [
    ("97098f0c828f841774abb63b5692fc0a6d7e9321", ["Cat", "Motya"], "[R] only Motya is in the frame, Moris is not visible"),
    ("29dfec973284b3557eadd7177063fef258a33166", [], "[R][I] placeholder with the text 'Adopted', no animal"),
    # [I] Cat label, but no cat in the photo -> label removed, kept as a background image
    ("09924b95d298044e5ae7ed0b1326e72c08d36be8", [], "[I] labelled Cat, no cat in the photo"),
    ("7658de36010e2a59e0e72c464577df45aea0f816", [], "[I] labelled Cat, no cat in the photo"),
    ("cd0a07e5a4f4572508383d591d78d4da2cdf3a57", [], "[I] labelled Cat, no cat in the photo"),
    ("7fe2f8f92d2ebfe96c4735e57e8e641c135a308b", [], "[I] labelled Cat, no cat in the photo"),
    # [I] Dog label, but no dog in the photo -> label removed, kept as a background image
    ("50a5e99c66688cd282d42dc1bc7623a2ce28429f", [], "[I] labelled Dog, no dog in the photo"),
    ("4de276cb03866d792873de49aade972c7e432104", [], "[I] labelled Dog, no dog in the photo"),
    ("32ed1303705d157fd1e414d32c5c6f47bf2fdfcd", [], "[I] labelled Dog, no dog in the photo"),
    ("4c693300a25630424712c54439be23e9485a1f1f", [], "[I] labelled Dog, no dog in the photo"),
    # [I] a dog in the photo, but no Dog label -> Dog added
    ("e9f14e9e3228d299a76deb656dbd6324684d87b2", ["Cat", "Dog"], "[I] dog in the photo, Dog label missing"),
    ("e5a95c08a393217b2ed651310e3bd68da0338f8e", ["Cat", "Dog"], "[I] dog in the photo, Dog label missing"),
]

# (image_id, reason): train images left out of training entirely, together with their train duplicates
DROP_IMAGES = [
    ("e6529dc1df879afab018f368f671fb7695ddcb9f", "[R][I] labelled Cat, but no cat is visible (small Santa photo on a white page)"),
]

ids = [f[0] for f in LABEL_FIXES] + [d[0] for d in DROP_IMAGES]
assert len(ids) == len(set(ids)), "an image_id is listed twice"

CHANGES = []   # one row per changed image, for the gallery below
train_clean = train.copy()
for image_id, new_labels, reason in LABEL_FIXES:
    match = train_clean.image_id == image_id
    assert match.sum() == 1, f"{image_id} not found in train"
    old_labels = label_names(train_clean[match].iloc[0], LABELS)
    rows = train_clean.dup_group == train_clean.dup_group[match].iloc[0]   # the image and its train duplicates
    train_clean.loc[rows, LABELS] = [int(label in new_labels) for label in LABELS]
    new_text = ", ".join(new_labels) or "none"
    CHANGES.append(dict(image_id=image_id, old=old_labels, new=new_text, reason=reason, copies=int(rows.sum())))

for image_id, reason in DROP_IMAGES:
    match = train_clean.image_id == image_id
    assert match.sum() == 1, f"{image_id} not found in train"
    rows = train_clean.dup_group == train_clean.dup_group[match].iloc[0]
    CHANGES.append(dict(image_id=image_id, old=label_names(train_clean[match].iloc[0], LABELS), new="DROPPED",
                        reason=reason, copies=int(rows.sum())))
    train_clean = train_clean[~rows]

CHANGES = pd.DataFrame(CHANGES)
display(CHANGES)

# Train-train duplicates: keep the first image of each duplicate group, drop the rest
is_dup = train_clean.dup_group.duplicated()
train_clean = train_clean[~is_dup].reset_index(drop=True)
print(f"Dropped {is_dup.sum()} train duplicates -> {len(train_clean)} train images")
display(pd.DataFrame({"before": train[LABELS].sum(), "after": train_clean[LABELS].sum()}))

train_clean.drop(columns="path").to_csv(OUT_DIR / "train_clean.csv", index=False)
print("saved", OUT_DIR / "train_clean.csv")

In [ ]:
# Every changed image: old labels -> new labels, so each fix can be confirmed by eye
shown = CHANGES.merge(train[["image_id", "path"]], on="image_id")
PER_ROW = 5
show_grid([shown.iloc[j:j + PER_ROW] for j in range(0, len(shown), PER_ROW)],
          title=lambda r: f"{r.old}  ->  {r.new}\n{r.reason[:45]}\n{r.image_id}",
          suptitle=f"All {len(shown)} label changes in train (old -> new)", ncols=PER_ROW, size=4)

## 11. Scene groups and the validation split
**Problem (shortcut learning):** all Moris / Motya / Biatrix photos come from one owner, often several shots of the same scene
(same bed, same snow field). If one shot is in train and a sibling shot in val, the model can score on val by recognising the *scene*
instead of the *pet*, and we would never notice.

**Fix for the validation:**
1. Group the rare-class photos into **scenes**: CNN features (ImageNet-pretrained `convnext_tiny`, frozen) + average-linkage clustering
   on cosine similarity. Only the ~125 rare-class images, so it takes seconds even on CPU.
2. **Grouped hold-out (20%)**: a scene goes to train or val as a whole, never split. The rest (Cat / Dog / background) is a normal
   stratified 20% split. Val then measures whether the model recognises the **pet in a scene it has never seen**.

Saved as `scene_group` and `is_val` in `train_clean.csv`, so every experiment uses the same split.
Note: test was split randomly from the same pool (duplicate statistics, section 6), so the leaderboard *does* contain sibling shots.
The grouped val is therefore a little stricter than the LB; the gap between them shows how much a model leans on the scene.

In [ ]:
import torch, timm
import torch.nn.functional as F
import torchvision.transforms as T
from scipy.cluster.hierarchy import linkage, fcluster

emb_model = timm.create_model("convnext_tiny.fb_in1k", pretrained=True, num_classes=0).eval()
cfg = emb_model.pretrained_cfg
emb_tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(cfg["mean"], cfg["std"])])   # whole image, no centre crop

is_rare = train_clean[rare].any(axis=1)
rare_imgs = train_clean[is_rare]
with torch.no_grad():
    feats = []
    for start in range(0, len(rare_imgs), 32):
        x = torch.stack([emb_tf(Image.open(p).convert("RGB")) for p in rare_imgs.path.iloc[start:start + 32]])
        feats.append(emb_model(x))
    rare_emb = F.normalize(torch.cat(feats), dim=1).numpy()   # dot product = cosine similarity
print("rare-class images:", len(rare_imgs), "| feature size:", rare_emb.shape[1])

### Tuning the scene threshold
`SCENE_SIM` is the main knob. The sweep shows what each value would give; the borderline pairs show photo pairs whose similarity is just above and
just below the current threshold, each titled with what the clustering actually did (`SAME scene` / `separate`, plus the scene name).
A pair's similarity alone does not decide it: average linkage compares a photo with **all** photos of a scene.
Look for sibling shots marked `separate` (threshold too strict) or different places marked `SAME scene` (too loose); then move `SCENE_SIM` and rerun the cells from here. `MANUAL_SAME_SCENE` forces photos into one scene when the clustering misses an obvious match
(grouping photos is allowed: it changes no label).

In [ ]:
# What each threshold would give: many singletons = too strict; one giant group per pet = too loose
Z = linkage(rare_emb, method="average", metric="cosine")
rows = []
for sim in [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    s = pd.Series(fcluster(Z, t=1 - sim, criterion="distance"))
    sz = s.value_counts()
    biggest = rare_imgs[rare].values[(s == sz.index[0]).values].sum(axis=0)   # rare positives in the largest scene
    rows.append(dict(SCENE_SIM=sim, scenes=len(sz), singletons=int((sz == 1).sum()), largest=int(sz.max()),
                     **{f"largest_{l}": int(b) for l, b in zip(rare, biggest)}))
display(pd.DataFrame(rows).set_index("SCENE_SIM"))
print("Rule of thumb: no scene should hold more than ~25% of one pet's photos, or that pet cannot be split 80/20 by scenes.")

In [ ]:
# Two photos are the same scene when their cosine similarity is >= SCENE_SIM (average linkage: no long chains).
# Check the galleries below: too low merges different scenes (or all photos of one pet), too high leaves sibling shots apart.
SCENE_SIM = 0.80
NUMBERING_SIM = 0.80   # FIXED: only defines the "#k" photo numbers in the per-pet galleries. Never change it, or the numbers
                       # in MANUAL_SAME_SCENE_BY_NUM point at different photos.
from scipy.cluster.hierarchy import leaves_list

Z_rare = linkage(rare_emb, method="average", metric="cosine")
scene = fcluster(Z_rare, t=1 - SCENE_SIM, criterion="distance")
train_clean["scene_group"] = "img_" + train_clean.index.astype(str)          # every other image is its own group
train_clean.loc[is_rare, "scene_group"] = ["scene_" + str(s) for s in scene]

# Frozen per-pet numbering: similarity order of each pet's photos, photos of one (automatic) scene kept together.
# PET_ORDER[pet][k] = image_id of photo #k in that pet's gallery.
auto_scene = fcluster(Z_rare, t=1 - NUMBERING_SIM, criterion="distance")
PET_ORDER = {}
for pet in rare:
    pos = np.where(rare_imgs[pet].values == 1)[0]
    leaf = pos[leaves_list(linkage(rare_emb[pos], method="average", metric="cosine", optimal_ordering=True))]
    o = pd.DataFrame({"image_id": rare_imgs.image_id.values[leaf], "scene": auto_scene[leaf], "leaf_rank": np.arange(len(leaf))})
    o["first"] = o.groupby("scene").leaf_rank.transform("min")
    PET_ORDER[pet] = o.sort_values(["first", "leaf_rank"]).image_id.tolist()

# Same scene by hand, using the "#k" numbers of the per-pet galleries (each list = one scene; overlapping lists are joined)
MANUAL_SAME_SCENE_BY_NUM = {
    "Moris":   [[1, 2, 3], [8, 9], [11, 12, 13, 24, 36, 46], [16, 17], [23, 24]],
    "Motya":   [[1, 2], [10, 11], [6, 7], [14, 15, 16]],
    "Biatrix": [[0, 1, 2], [4, 5], [9, 10], [22, 23, 24], [28, 29], [30, 31, 32, 33, 34, 35, 36]],
}
# Same scene by hand, using image_ids directly (each list = one scene)
MANUAL_SAME_SCENE = [
    # ["<image_id 1>", "<image_id 2>", ...],
]

manual = [[PET_ORDER[pet][k] for k in nums] for pet, lists in MANUAL_SAME_SCENE_BY_NUM.items() for nums in lists]
for ids in manual + MANUAL_SAME_SCENE:
    hit = train_clean.image_id.isin(ids)
    assert hit.sum() == len(set(ids)), f"not found: {set(ids) - set(train_clean.image_id[hit])}"
    groups = train_clean.loc[hit, "scene_group"].unique()
    train_clean.loc[train_clean.scene_group.isin(groups), "scene_group"] = groups[0]   # merge their whole scenes

sizes = train_clean[is_rare].scene_group.value_counts()
print(f"{len(manual) + len(MANUAL_SAME_SCENE)} manual merges applied")
print(f"{len(sizes)} scenes from {is_rare.sum()} rare-class images | singletons: {(sizes == 1).sum()} | "
      f"multi-photo scenes: {(sizes > 1).sum()} | largest: {sizes.max()} photos")
display(train_clean[is_rare].groupby("scene_group")[rare].sum().loc[sizes.index].head(15))

In [ ]:
# All scenes, one gallery PER RARE CLASS, for checking and grouping scenes by hand.
# Each grid row = one scene as the split uses it (automatic clustering + manual merges). "#k" numbers are FROZEN (PET_ORDER,
# clustering cell): a photo keeps its number after merges, so you can keep adding lists to MANUAL_SAME_SCENE_BY_NUM.
# Scenes are ordered by their lowest number; "sim prev" = similarity to the previous photo in this gallery.
# Photos with two pets appear in both galleries (marked "2 pets").
PER_ROW, ROWS_PER_FIG = 6, 8
emb_row = pd.Series(np.arange(len(rare_imgs)), index=rare_imgs.image_id.values)

for pet in rare:
    pet_df = pd.DataFrame({"image_id": PET_ORDER[pet], "k": np.arange(len(PET_ORDER[pet]))})
    pet_df = pet_df.merge(train_clean[["image_id", "scene_group", *LABELS, "path"]], on="image_id").rename(columns={"scene_group": "scene"})
    pet_df["n_pets"] = pet_df[rare].sum(axis=1)
    pet_df["first"] = pet_df.groupby("scene").k.transform("min")
    pet_df = pet_df.sort_values(["first", "k"]).reset_index(drop=True)
    e = rare_emb[emb_row[pet_df.image_id].values]
    pet_df["sim_prev"] = np.r_[np.nan, np.sum(e[1:] * e[:-1], axis=1)]

    print(f"\n===== {pet}: {len(pet_df)} photos in {pet_df.scene.nunique()} scenes =====")
    for scene_name, g in pet_df.groupby("scene", sort=False):
        print(f"--- {scene_name} ({len(g)} photos): #" + ", #".join(map(str, g.k)))
        for _, r in g.iterrows():
            gap = "" if np.isnan(r.sim_prev) else f"{r.sim_prev:.2f}"
            print(f"  #{r.k:3d}  sim prev {gap:>4}  {r.image_id}  {pet_labels(r)}" + ("  (2 pets)" if r.n_pets > 1 else ""))

    # Grid rows: one per scene (long scenes wrap onto extra rows)
    rows = [g.iloc[j:j + PER_ROW] for _, g in pet_df.groupby("scene", sort=False) for j in range(0, len(g), PER_ROW)]
    for f in range(0, len(rows), ROWS_PER_FIG):
        show_grid(rows[f:f + ROWS_PER_FIG],
                  title=lambda r: f"#{r.k} | {r.scene}" + ("" if np.isnan(r.sim_prev) else f" | sim prev {r.sim_prev:.2f}")
                                  + ("\n" + pet_labels(r) + (" (2 pets)" if r.n_pets > 1 else "")) + f"\n{r.image_id}",
                  suptitle=f"{pet}: one row per scene (rows {f + 1}-{min(f + ROWS_PER_FIG, len(rows))} of {len(rows)})",
                  ncols=PER_ROW, size=3.6)

In [ ]:
# Every multi-photo scene as one row (up to 8 photos): are these really the same scene?
multi = sizes[sizes > 1].index
show_grid([train_clean[train_clean.scene_group == g].head(8).assign(g=g) for g in multi],
          title=lambda r: f"{r.g} | {pet_labels(r)}\n{r.image_id[:12]}",
          suptitle=f"Rare-class scenes with more than one photo (SCENE_SIM = {SCENE_SIM})", ncols=8)

In [ ]:
VAL_FRAC = 0.2
rng = np.random.RandomState(SEED)

# 1) Rare-class images: whole scenes go to val. Random search for the scene assignment whose val share
#    is closest to VAL_FRAC for every rare class at once.
G = train_clean[is_rare].groupby("scene_group")[rare].sum()   # positives per scene
target = G.sum() * VAL_FRAC
best_err, best_pick = np.inf, None
for _ in range(20000):
    pick = rng.rand(len(G)) < VAL_FRAC
    err = (np.abs(G[pick].sum() - target) / target).sum()
    if err < best_err:
        best_err, best_pick = err, pick
val_scenes = set(G.index[best_pick])

# 2) All other images (only Cat / Dog / background patterns are left): ordinary stratified split on the label pattern
from sklearn.model_selection import train_test_split
rest = train_clean[~is_rare]
pattern = rest[LABELS].astype(str).agg("".join, axis=1)
pattern = pattern.where(pattern.map(pattern.value_counts()) >= 2, "other")
_, rest_val = train_test_split(rest.index, test_size=VAL_FRAC, stratify=pattern, random_state=SEED)

train_clean["is_val"] = train_clean.scene_group.isin(val_scenes) | train_clean.index.isin(rest_val)
assert train_clean.groupby("scene_group").is_val.nunique().max() == 1, "a scene is split across train and val"

split = train_clean.assign(none=(train_clean[LABELS].sum(axis=1) == 0).astype(int)).groupby("is_val")[LABELS + ["none"]].sum()
split.index = split.index.map({False: "train", True: "val"})
split.loc["val_%"] = (split.loc["val"] / split.sum() * 100).round(1)
split.insert(0, "images", [(~train_clean.is_val).sum(), train_clean.is_val.sum(), round(train_clean.is_val.mean() * 100, 1)])
display(split)
print(f"rare-class scenes: {len(G)} | in val: {len(val_scenes)}")

train_clean.drop(columns="path").to_csv(OUT_DIR / "train_clean.csv", index=False)
print("saved", OUT_DIR / "train_clean.csv", "with scene_group and is_val")

## 12. Summary & modelling decisions

### What the data looks like
| | Finding | Numbers |
|---|---|---|
| Size | train 12,260 / test 18,673 (test is 1.5× train); **every image is 450×375** | 1 distinct size in both sets |
| Labels | multi-label, 5 binary columns, max 4 labels per image | |
| Imbalance | Cat 41.4%, Dog 41.0%, **Moris 56, Motya 42, Biatrix 41** positives (0.33–0.46%, neg:pos ≈ 220–300) | |
| Background | **17.6% (2,162) images have no label**: other animals (insects, birds, sea life, …), not empty scenes | |
| Hierarchy | Moris/Motya always come with Cat, Biatrix always with Dog. Cat+Dog together only in 4 images (`11101`, the owner's pets) | 8 label patterns in total |
| Rare classes | one owner's phone photos at home: Moris = black long-haired cat, Motya = calico, Biatrix = corgi, **often small / far away** (snow, grass) | |
| Quality shift | brightness / contrast / saturation / sharpness distributions **match** between train and test (means differ by < 0.01) | |
| Odd images | ~1% grayscale, ~0.3–0.5% very dark, ~0.8% very bright in **both** sets; **1 blank white test image** (`d20a3fd2`) with no train equivalent | IQR outliers: train 3, test *see 7* |
| Duplicates | train–train 162 pairs (324 images), train–test 430, test–test 302; max group size 2; **0 label conflicts** | |
| Label noise | wrong train labels fixed and unusable train images dropped, found by the rare-class review (section 8) and the ImageNet label check (section 9) (section 10; train relabelling confirmed as allowed by the competition owner) | |

### Decisions
1. **Metric & thresholds:** the score is **macro F1**, so Moris, Motya and Biatrix are 60% of it, even though they have only ~140 train images
   and roughly 60–85 expected positives each in test (at train prevalence). One missed or false rare prediction moves the score far more than many Cat/Dog errors.
   Predict probabilities, tune **one threshold per class** on out-of-fold predictions, then **enforce the hierarchy**
   (Moris/Motya → Cat, Biatrix → Dog), which holds in 100% of train. Submit the thresholded 0/1 values, matching `sample_submission.csv`.
2. **Validation: grouped hold-out (section 11).** 20% of train, fixed in `train_clean.csv` (`is_val`). Rare-class photos are grouped into
   **scenes** and a scene is never split between train and val, so val measures recognising the pet in a scene it has not seen
   (the shortcut check). Cat / Dog / background: ordinary stratified split. One model per experiment keeps compute low;
   the val score is noisy for the rare classes (~8–11 positives each), so only trust clear differences. Report val next to the Public LB:
   the LB contains sibling shots, so LB > val by a large margin means the model leans on the scene.
3. **Train–train duplicates: drop them** (keep one per group, see section 10). They hurt little: 162 extra copies (1.3%) barely change training.
   But they leak when the two copies land in different folds: ~1/3 of pairs at 80/20, so ~50 val images the model has already seen, which inflates CV and skews thresholds.
   They are exact copies with consistent labels, so dropping them loses no information. (Equivalent alternative: keep them and use `dup_group` as the group in a grouped split.)
4. **Train–test duplicates (430, 2.3% of test):** proof that the leaderboard partly rewards memorisation. Nothing to fix; useful as a sanity check
   (predictions on these should match the train labels).
5. **Backbone:** any `timm` weights are allowed, so stronger pretrained backbones (e.g. ImageNet-21k / larger-pretraining
   ConvNeXt, EfficientNetV2, EVA) are fair game. Fine-grained identity classes with ~50 examples benefit most from better features.
6. **Resolution:** all images are 450×375, so one fixed resize with no letterboxing. The identity classes are fine-grained and Biatrix is often tiny,
   so use **high resolution** (384, or **384×320 to keep the 1.2 aspect ratio** and avoid extra squashing).
7. **Augmentations:** train and test look the same, so augmentations are **not needed to cover a shift**. Their only job is **regularisation**,
   mainly against overfitting the ~50 rare-class images.
   - Geometric: **horizontal flip** yes. **Mild** crop/scale only (keep ≥ 85% of each side): strong crops cut out a small corgi or one of two cats while the label stays 1.
     No vertical flip or rotation (photos are upright).
   - Photometric: mild ColorJitter, RandomGrayscale p≈0.05 (~1% grayscale in both sets), light blur p≈0.1. Expect a small effect; check it with an ablation, don't assume.
   - Mixup is fine for multi-label (mix the labels too). CutMix is risky, because the patch can remove the labelled animal.
8. **Rare classes are where the score is decided:** the bigger levers are the **loss / sampling** (pos_weight, oversampling, focal/ASL) and the thresholds.
   **Shortcut risk:** all rare positives come from one owner, so the model can learn the room / snow field instead of the pet. Fight it in training
   (scene-changing augmentations, oversampling rare images with fresh augmentations), detect it with the grouped val (section 11),
   GradCAM on rare-class val images and the false-positive gallery (ordinary cats / dogs predicted as Moris / Motya / Biatrix).
9. **Background images** stay in training as all-zero targets; the model must be allowed to output all zeros.
10. **Blank test image:** no manual override (test is never relabelled). The ~2,000 background images in train already teach
    the model to output all zeros for "nothing of interest", so just check what it predicts there.